# SHAP Drift Analysis: Hardware-Induced Truncation

This notebook investigates how different smartphone hardware alters the acoustic features (data drift) and how the XGBoost regressor corrects for this bias. We specifically focus on the F0 (fundamental frequency) variability truncation caused by smartphone noise-cancellation algorithms.

In [ ]:
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic parameters
sns.set_theme(style="whitegrid")

## 1. Load Data and Model

In [ ]:
# Define features used in the model
features = [
    'f0_mean_noisy', 'jitter_local_noisy', 'shimmer_local_noisy', 
    'F1_mean_noisy', 'F2_mean_noisy', 'sex', 'smartphone_model', 'task'
]

# Attempt to load model and data
try:
    model = xgb.XGBRegressor()
    model.load_model("../models/xgboost_debias.json")
    print("Model loaded successfully.")
    
    df = pd.read_parquet("../data/processed/ml_dataset.parquet")
    
    # Convert categoricals
    categorical_cols = ['sex', 'smartphone_model', 'task']
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    df_clean = df.dropna(subset=features + ['f0_mean_clean']).copy()
    X = df_clean[features]
    print(f"Data loaded successfully. Analyzed events: {len(X)}")
except Exception as e:
    print(f"Dataset/Model not fully available in this environment. Error: {e}")
    # Creating dummy data for demonstration if needed, but in production it relies on actual data
    print("Please ensure the ML pipeline has run to generate the processed dataset.")


## 2. Compute SHAP Values

SHAP (SHapley Additive exPlanations) values break down a prediction to show the impact of each feature. We use TreeExplainer optimized for XGBoost.

In [ ]:
if 'X' in locals() and len(X) > 0:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X)
    print("SHAP values computed successfully.")


## 3. Global Feature Importance (SHAP Summary Plot)

The summary plot ranks the most important features driving the XGBoost predictions. It also shows whether high or low values of a feature positively or negatively impact the prediction.

In [ ]:
if 'shap_values' in locals():
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X, show=False)
    plt.title("SHAP Summary Plot: Impact of Noisy Features on Corrected Baseline")
    plt.tight_layout()
    plt.savefig("../output/plots/shap_summary.png", dpi=300)
    plt.show()


## 4. Demonstrating Hardware Drift (SHAP Dependence Plot)

The dependence plot highlights how the interaction between features affects the model. For instance, we can visualize how the hardware (smartphone model) truncates F0 variability. Notice how the corrections applied by the model differ based on both the `f0_mean_noisy` and the underlying `smartphone_model`.

In [ ]:
if 'shap_values' in locals():
    # If we have F0 mean noisy, let's plot its dependence
    if 'f0_mean_noisy' in X.columns:
        plt.figure(figsize=(8, 6))
        # We interact with smartphone_model to show the hardware effect
        interaction_feature = 'smartphone_model' if 'smartphone_model' in X.columns else 'auto'
        shap.dependence_plot("f0_mean_noisy", shap_values.values, X, interaction_index=interaction_feature, show=False)
        plt.title("SHAP Dependence Plot: Hardware Interaction with F0")
        plt.tight_layout()
        plt.savefig("../output/plots/shap_dependence_f0.png", dpi=300)
        plt.show()
